# Tekrarlayan sinir ağları

Önceki modülde, metnin zengin anlamsal temsillerini ele aldık. Kullandığımız mimari, bir cümledeki kelimelerin toplanmış anlamını yakalar, ancak kelimelerin **sırasını** hesaba katmaz, çünkü gömme işlemini takiben yapılan toplama işlemi, bu bilgiyi orijinal metinden kaldırır. Bu modeller kelime sırasını temsil edemediği için metin oluşturma veya soru yanıtlama gibi daha karmaşık veya belirsiz görevleri çözemezler.

Bir metin dizisinin anlamını yakalamak için, **tekrarlayan sinir ağı** (recurrent neural network, RNN) adı verilen bir sinir ağı mimarisi kullanacağız. Bir RNN kullanırken, cümlemizi bir seferde bir token geçiririz ve ağ bir **durum** üretir; daha sonra bu durumu, sonraki token ile birlikte ağa tekrar veririz.

![Bir örnek tekrarlayan sinir ağı üretimini gösteren görsel.](../../../../../translated_images/tr/rnn.27f5c29c53d727b5.webp)

$X_0,\dots,X_n$ token girdi dizisi verildiğinde, RNN bir sinir ağı blokları dizisi oluşturur ve bu diziyi uçtan uca geriye yayılım (backpropagation) ile eğitir. Her ağ bloğu $(X_i,S_i)$ çiftini girdi olarak alır ve sonuç olarak $S_{i+1}$ üretir. Son durum $S_n$ veya çıktı $Y_n$ bir doğrusal sınıflandırıcıya girer ve sonuç üretilir. Tüm ağ blokları aynı ağırlıkları paylaşır ve bir geriye yayılım üzerinden uçtan uca eğitilir.

> Yukarıdaki şekil, tekrarlayan sinir ağını açılmış haliyle (solda) ve daha kompakt tekrarlayan temsiliyle (sağda) gösterir. Tüm RNN hücrelerinin aynı **paylaşılabilir ağırlıklara** sahip olduğunu anlamak önemlidir.

Durum vektörleri $S_0,\dots,S_n$ ağ boyunca aktarıldığı için, RNN kelimeler arasındaki ardışık bağımlılıkları öğrenebilir. Örneğin, dizide herhangi bir yerde *not* kelimesi geçtiğinde, durum vektöründeki belirli öğeleri olumsuzlamayı öğrenebilir.

İçeride, her RNN hücresi iki ağırlık matrisi içerir: $W_H$ ve $W_I$, ve bir bias $b$. Her RNN adımında, girdi $X_i$ ve girdi durumu $S_i$ verildiğinde, çıktı durumu $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ olarak hesaplanır; burada $f$ bir aktivasyon fonksiyonudur (genellikle $\tanh$).

> Metin oluşturma (bir sonraki bölümde ele alacağız) veya makine çevirisi gibi problemler için her RNN adımında bir çıktı değeri almak isteriz. Bu durumda, ayrıca başka bir matris $W_O$ vardır ve çıktı $Y_i=f(W_O\times S_i+b_O)$ olarak hesaplanır.

Tekrarlayan sinir ağlarının haber veri setimizi sınıflandırmamıza nasıl yardımcı olabileceğine bakalım.

> Sandbox ortamı için, gerekli kütüphanenin yüklendiğinden ve verinin önceden getirildiğinden emin olmak için aşağıdaki hücreyi çalıştırmamız gerekir. Yerel olarak çalıştırıyorsanız, aşağıdaki hücreyi atlayabilirsiniz.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Büyük modeller eğitilirken, GPU bellek tahsisi bir sorun haline gelebilir. Ayrıca, verilerin GPU belleğimize sığması ve eğitimin yeterince hızlı olması için farklı minibatch boyutlarıyla deneme yapmamız gerekebilir. Bu kodu kendi GPU makinenizde çalıştırıyorsanız, eğitimi hızlandırmak için minibatch boyutunu ayarlamayı deneyebilirsiniz.

> **Not**: NVidia sürücülerinin belirli sürümlerinin modeli eğittikten sonra belleği serbest bırakmadığı bilinmektedir. Bu not defterinde birkaç örnek çalıştırıyoruz ve özellikle aynı not defteri içinde kendi deneylerinizi yapıyorsanız, belirli kurulumlarda belleğin tükenmesine neden olabilir. Modeli eğitmeye başlarken garip hatalarla karşılaşırsanız, not defteri çekirdeğini yeniden başlatmak isteyebilirsiniz.


In [3]:
batch_size = 16
embed_size = 64

## Basit RNN sınıflandırıcısı

Basit RNN durumunda, her tekrarlayan birim, bir giriş vektörü ve durum vektörü alan ve yeni bir durum vektörü üreten basit bir lineer ağdır. Keras'ta bu, `SimpleRNN` katmanı ile temsil edilebilir.

One-hot kodlanmış tokenları doğrudan RNN katmanına iletebiliriz, ancak yüksek boyutsallıklarından dolayı bu iyi bir fikir değildir. Bu nedenle, kelime vektörlerinin boyutsallığını düşürmek için bir embedding katmanı, ardından bir RNN katmanı ve son olarak bir `Dense` sınıflandırıcısı kullanacağız.

> **Not**: Boyutsallığın çok yüksek olmadığı durumlarda, örneğin karakter düzeyinde tokenizasyon kullanıldığında, one-hot kodlanmış tokenları doğrudan RNN hücresine iletmek mantıklı olabilir.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Not:** Burada basitlik için eğitilmemiş bir gömme katmanı kullanıyoruz, ancak daha iyi sonuçlar için önceki birimde açıklandığı gibi Word2Vec kullanarak önceden eğitilmiş bir gömme katmanı kullanabiliriz. Bu kodu önceden eğitilmiş gömme katmanlarıyla çalışacak şekilde uyarlamak sizin için iyi bir alıştırma olur.

Şimdi RNN'imizi eğitelim. Genel olarak RNN'lerin eğitimi oldukça zordur, çünkü RNN hücreleri dizinin uzunluğu boyunca açığa çıkarıldığında, geri yayılımda yer alan katman sayısı oldukça fazladır. Bu nedenle daha küçük bir öğrenme oranı seçmeli ve iyi sonuçlar elde etmek için ağı daha büyük bir veri setinde eğitmeliyiz. Bu oldukça uzun zaman alabilir, bu yüzden GPU kullanmak tercih edilir.

İşleri hızlandırmak için, açıklamayı atlayarak sadece haber başlıkları üzerinde RNN modelini eğiteceğiz. Açıklama ile eğitimi deneyebilir ve modelin eğitilip eğitilemediğini görebilirsiniz.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Not** burada doğruluk muhtemelen daha düşük olacaktır, çünkü yalnızca haber başlıkları üzerinde eğitim yapıyoruz.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Artık maskelenmeyi kullandığımıza göre, modeli tüm başlıklar ve açıklamalar veri kümesi üzerinde eğitebiliriz.

> **Not**: Haber başlıklarında eğitilmiş bir vektörleştirici kullandığımızı ve makalenin tüm metnini değil, fark ettiniz mi? Bu, bazı tokenlerin göz ardı edilmesine neden olabilir, bu yüzden vektörleştiriciyi yeniden eğitmek daha iyidir. Ancak, bunun etkisi çok küçük olabilir, bu yüzden basitlik adına önceki önceden eğitilmiş vektörleştiriciyi kullanmaya devam edeceğiz.


## LSTM: Uzun kısa süreli hafıza

RNN'lerin ana problemlerinden biri **gradyanların kaybolmasıdır**. RNN'ler oldukça uzun olabilir ve geri yayılım sırasında gradyanları ağın ilk katmanına kadar taşımakta zorlanabilir. Bu olduğunda, ağ uzak tokenlar arasındaki ilişkileri öğrenemez. Bu problemi önlemenin bir yolu, **kapılar** kullanarak **açık durum yönetimi** sağlamaktır. Kapılar getiren en yaygın iki mimari **uzun kısa süreli hafıza** (LSTM) ve **kapılı röle birimi** (GRU)dur. Burada LSTM'leri ele alacağız.

![Uzun kısa süreli hafıza hücresine örnek gösteren görsel](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Bir LSTM ağı, RNN'ye benzer şekilde organize edilmiştir, ancak katmanlar arasında geçen iki durum vardır: gerçek durum $c$ ve gizli vektör $h$. Her bir ünitede, gizli vektör $h_{t-1}$ giriş $x_t$ ile birleştirilir ve birlikte durum $c_t$ ile çıkış $h_t$ üzerinde ne olacağını **kapılar** aracılığıyla kontrol ederler. Her kapının sigmoid aktivasyonu vardır (çıkışı $[0,1]$ aralığında), bu durum vektörüyle çarpıldığında bit bazında bir maske gibi düşünülebilir. LSTM'lerin şu kapıları vardır (yukarıdaki resimde soldan sağa doğru):
* Vektör $c_{t-1}$'in hangi bileşenlerini unutmamız gerektiğini, hangilerini geçireceğimizi belirleyen **unutma kapısı**.
* Girdi vektörü ve önceki gizli vektörden ne kadar bilginin durum vektörüne dahil edileceğini belirleyen **girdi kapısı**.
* Yeni durum vektörünü alarak hangi bileşenlerinin yeni gizli vektör $h_t$'nin üretilmesinde kullanılacağını belirleyen **çıkış kapısı**.

Durum $c$'nin bileşenleri açılıp kapatılabilen bayraklar olarak düşünülebilir. Örneğin, dizide *Alice* ismine rastladığımızda bunun bir kadın ismi olduğunu tahmin eder ve cümlede kadın isim olduğunu belirten bayrağı kaldırırız. Sonrasında *and Tom* kelimelerine geldikçe çoğul isim olduğunu belirten bayrağı da kaldırırız. Böylece durumu manipüle ederek cümlenin dilbilgisel özelliklerini takip edebiliriz.

> **Not**: LSTM iç yapısını anlamak için harika bir kaynak: Christopher Olah'ın [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

Bir LSTM hücresinin iç yapısı karmaşık görünebilir, ancak Keras bu uygulamayı `LSTM` katmanı içinde gizler, bu yüzden yukarıdaki örnekte yapmamız gereken tek şey rekürrent katmanı değiştirmektir:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Not:** LSTM eğitiminin de oldukça yavaş olduğunu unutmayın ve eğitim başlangıcında doğrulukta çok fazla artış görmeyebilirsiniz. İyi bir doğruluk elde etmek için bir süre eğitime devam etmeniz gerekebilir.


## Çift Yönlü ve Çok Katmanlı RNN'ler

Şimdiye kadar örneklerimizde, tekrarlayan ağlar bir dizinin başından sonuna kadar çalıştı. Bu bizim için doğal geliyor çünkü konuşmayı dinlediğimiz veya okuduğumuz yönle aynı yönde ilerliyor. Ancak, giriş dizisine rastgele erişim gerektiren senaryolar için, tekrarlayan hesabı her iki yönde çalıştırmak daha mantıklıdır. Hesaplamalara her iki yönde izin veren RNN'lere **çift yönlü** RNN'ler denir ve bu, tekrarlayan katmanı özel bir `Bidirectonal` katmanı ile sararak oluşturulabilir.

> **Not**: `Bidirectional` katmanı, içindeki katmandan iki kopya oluşturur ve bu kopyalardan birinin `go_backwards` özelliğini `True` olarak ayarlar; böylece dizide ters yönde çalışmasını sağlar.

Tek yönlü veya çift yönlü olsun, tekrarlayan ağlar bir dizideki desenleri yakalar ve bunları durum vektörlerinde saklar ya da çıktı olarak döner. Konvolüsyonel ağlarda olduğu gibi, daha yüksek seviyeli desenleri yakalamak için ilk katmanın ardından başka bir tekrarlayan katman oluşturabiliriz; bu yüksek seviyeli desenler, ilk katman tarafından çıkartılan daha düşük seviyeli desenlerden oluşur. Bu da bizi, önceki katmanın çıktısının sonraki katmana girdi olarak aktarıldığı iki veya daha fazla tekrarlayan ağdan oluşan **çok katmanlı RNN** kavramına götürür.

![Çok Katmanlı uzun-kısa süreli bellek- RNN gösteren Görsel](../../../../../translated_images/tr/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Fotoğraf Fernando López'ün [bu harika gönderisinden](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3).*

Keras, bu ağları kurmayı kolaylaştırır, çünkü modele daha fazla tekrarlayan katman eklemeniz yeterlidir. Son katman hariç tüm katmanlar için `return_sequences=True` parametresini belirtmemiz gerekir, çünkü katmanın tüm ara durumları döndürmesini isteriz, sadece tekrarlayan hesabın son durumunu değil.

Sınıflandırma problemimiz için iki katmanlı çift yönlü bir LSTM oluşturalım.

> **Not** bu kodun tamamlanması yine oldukça uzun sürüyor, ancak şimdiye kadar gördüğümüz en yüksek doğruluğu sağlıyor. Belki de beklemeye ve sonucu görmeye değer.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## Diğer görevler için RNN'ler

Şimdiye kadar, RNN'leri metin dizilerini sınıflandırmak için kullanmaya odaklandık. Ancak metin üretimi ve makine çevirisi gibi çok daha fazla görevi gerçekleştirebilirler — bu görevleri bir sonraki birimde ele alacağız.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
